<a href="https://colab.research.google.com/github/priyal6/General-llm/blob/main/tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
from google.colab import userdata
import os

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


In [18]:
from langchain.tools import tool


@tool
def order_lookup(order_id: str) -> str:
  """Look up the status of an existing customer order."""

  orders = {
      "12345": "Shipped",
      "99999": "Processing"
      }
  if order_id not in orders:
    raise ValueError(f"Order {order_id} does not exist")

  return orders[order_id]

In [19]:
@tool
def calculator(expression: str) -> str:
  """Calculate a mathematical expression.

    Use this tool for arithmetic calculations including
    percentages, multiplication, division, addition,
    and subtraction.
    """

  return str(eval(expression))

In [20]:
@tool
def refund_policy_search(query: str) -> str:
    """Search the company's refund and returns policy.

    Use this tool for questions about refunds, returns,
    return windows, exchanges, or refund eligibility.
    """

    return """
    Products can normally be returned within 30 days.
    Items must be unused and in their original packaging.
    """

In [21]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

@wrap_tool_call
def handle_tool_error(request, handler):

    try:
        return handler(request)

    except ValueError as e:
        return ToolMessage(
            content=f"Tool error: {str(e)}",
            tool_call_id=request.tool_call["id"]
        )


In [34]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

agent = create_agent(
    model=ChatOpenAI(model_name="gpt-3.5-turbo", api_key=OPENAI_API_KEY),
    tools=[
        calculator,
        order_lookup,
        refund_policy_search
    ],
    middleware=[
        handle_tool_error
    ],
    system_prompt="You are a helpful assistant. Be concise and accurate",
)


In [24]:
result = agent.invoke({
    "messages": "Where is order 55555?"
})

In [29]:
for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Where is order 55555?
================================== Ai Message ==================================
Tool Calls:
  order_lookup (call_1TXjjvOMHJkvjbJWMl9GvjfO)
 Call ID: call_1TXjjvOMHJkvjbJWMl9GvjfO
  Args:
    order_id: 55555
================================= Tool Message =================================

Tool error: Order 55555 does not exist
================================== Ai Message ==================================

I could not find any information about order 55555. It seems that the order does not exist. If you have any other questions or need assistance with a different order, feel free to let me know!


Tool Retry

In [40]:
import random
from langchain.tools import tool

@tool
def shipping_api(order_id: str) -> str:
    """Get live shipping status for an order."""

    if random.random() < 0.7:
        raise ConnectionError(
            "Shipping provider temporarily unavailable"
        )

    return f"Order {order_id} is out for delivery."

In [41]:
from langchain.agents.middleware import ToolRetryMiddleware

agent = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[
        calculator,
        order_lookup,
        shipping_api,
        refund_policy_search
    ],
    middleware=[
        ToolRetryMiddleware(
            max_retries=3
        )
    ],
    system_prompt="You are a helpful assistant.Be concise and accurate",
)

In [42]:
results = agent.invoke({
    "messages": "What is the order shipping status with order id 12345"
})

In [43]:
for message in results["messages"]:
  message.pretty_print()

================================ Human Message =================================

What is the order shipping status with order id 12345
================================== Ai Message ==================================
Tool Calls:
  shipping_api (call_pGxEgRC6ysC9s99444x6ykOB)
 Call ID: call_pGxEgRC6ysC9s99444x6ykOB
  Args:
    order_id: 12345
================================= Tool Message =================================
Name: shipping_api

Tool 'shipping_api' failed after 4 attempts with ConnectionError: Shipping provider temporarily unavailable. Please try again.
================================== Ai Message ==================================

The shipping provider is currently temporarily unavailable, and I'm unable to retrieve the shipping status for order ID 12345 at the moment. Please try again shortly. Is there anything else I can assist you with?
